# 19 · Data Quality, Data Drift y Concept Drift

En producción, un modelo puede fallar aunque el código no cambie. Cambian fuentes, definiciones, comportamiento, políticas, sensores y poblaciones. Este lab trata el sistema como algo vivo.

## Objetivos
- Definir contracts de datos y validaciones.
- Detectar schema drift y cambios de distribución.
- Calcular PSI, KS, Jensen-Shannon y métricas categóricas.
- Diferenciar covariate shift, label shift y concept drift.
- Construir monitoreo por ventanas y segmentos.
- Diseñar alertas y estrategias de retraining.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import jensenshannon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
SEED=42; rng=np.random.default_rng(SEED)
def make_batch(n=3000,shift=0,concept=0):
 age=rng.normal(45+shift,12,n); income=rng.lognormal(10.5+shift*.01,.6,n); region=rng.choice(['N','C','S'],n,p=[.2,.55,.25])
 logit=-3+.04*(age-45)+.000015*income+(region=='C')*.5+concept*(age-45)/10
 y=rng.binomial(1,1/(1+np.exp(-logit)))
 return pd.DataFrame({'age':age,'income':income,'region':region,'y':y})
base=make_batch(); current=make_batch(shift=5); concept_batch=make_batch(concept=.7)

## 1. Data contracts
Antes de estadística, valida invariantes:
- columnas requeridas;
- tipos;
- rangos;
- unicidad;
- null rate;
- categorías permitidas;
- relaciones lógicas (`fecha_fin>=fecha_inicio`);
- freshness y volumen.

Herramientas: Great Expectations, Pandera, dbt tests, Deequ. Un schema válido no garantiza distribución estable, pero evita muchos incidentes.


In [ ]:
def validate(df):
 checks={
  'columns':set(['age','income','region','y']).issubset(df.columns),
  'age_range':df.age.between(0,120).all(),
  'income_nonnegative':df.income.ge(0).all(),
  'null_rate_lt_5pct':df.isna().mean().max()<.05,
  'region_allowed':df.region.isin(['N','C','S']).all(),
 }
 return pd.Series(checks)
print(validate(base))

## 2. PSI (Population Stability Index)
Compara proporciones en bins entre referencia y actual:
$$PSI=\sum_i(a_i-e_i)\ln(a_i/e_i)$$
Es popular en riesgo, pero umbrales como 0.1/0.25 son reglas heurísticas, no leyes universales. Depende de tamaño, binning y contexto.


In [ ]:
def psi(expected,actual,bins=10):
 cuts=np.quantile(expected,np.linspace(0,1,bins+1)); cuts[0],cuts[-1]=-np.inf,np.inf
 e=np.histogram(expected,bins=cuts)[0]/len(expected); a=np.histogram(actual,bins=cuts)[0]/len(actual); e=np.clip(e,1e-8,None); a=np.clip(a,1e-8,None)
 return float(np.sum((a-e)*np.log(a/e)))
print('PSI age',psi(base.age,current.age)); print('PSI income',psi(base.income,current.income))

## 3. KS y otras distancias
KS compara CDFs y entrega estadístico/p-value. Con datasets enormes, diferencias diminutas serán 'significativas'; complementa con effect size. Jensen-Shannon mide diferencia entre distribuciones discretas y es simétrica/acotada. Wasserstein tiene interpretación de distancia de transporte.


In [ ]:
for c in ['age','income']:
 ks,p=stats.ks_2samp(base[c],current[c]); w=stats.wasserstein_distance(base[c],current[c]); print(c,'KS',ks,'p',p,'Wasserstein',w)
# categórica: total variation distance
def tv_cat(a,b):
 cats=sorted(set(a)|set(b)); pa=pd.Series(a).value_counts(normalize=True).reindex(cats,fill_value=0); pb=pd.Series(b).value_counts(normalize=True).reindex(cats,fill_value=0); return .5*np.abs(pa-pb).sum()
print('region TV',tv_cat(base.region,current.region))

## 4. Covariate shift vs concept drift
- **Covariate/data drift:** cambia $P(X)$.
- **Label shift:** cambia $P(Y)$.
- **Concept drift:** cambia $P(Y|X)$; el mismo perfil ahora tiene otra relación con el target.

Podemos detectar $P(X)$ sin labels; para concept drift normalmente necesitamos outcomes atrasados o señales proxy.


In [ ]:
# entrenar en base y comparar AUC en batches con/ sin concept drift
def prep(df): return pd.get_dummies(df[['age','income','region']],columns=['region']).reindex(columns=['age','income','region_C','region_N','region_S'],fill_value=0)
m=LogisticRegression(max_iter=3000).fit(prep(base),base.y)
for name,d in [('base',base),('covariate_shift',current),('concept_drift',concept_batch)]:
 p=m.predict_proba(prep(d))[:,1]; print(name,'prevalence',d.y.mean(),'AUC',roc_auc_score(d.y,p),'mean_score',p.mean())

## 5. Drift por segmentos
Un promedio nacional puede ocultar degradación en una región, dispositivo, sexo, edad o canal. Monitorea segmentos relevantes, pero controla ruido estadístico y privacidad.


In [ ]:
for region,g in current.groupby('region'):
 p=m.predict_proba(prep(g))[:,1]; print(region,'n',len(g),'AUC',roc_auc_score(g.y,p) if g.y.nunique()>1 else np.nan)

## 6. Performance delay
En fraude o abandono, la etiqueta real puede llegar semanas/meses después. Conviene distinguir:
- métricas inmediatas: drift, score distribution, rate de alertas;
- métricas con delay: precision/recall/outcome;
- backfill cuando aparecen labels.

## 7. ¿Cuándo reentrenar?
Opciones: calendario, suficiente data nueva, performance gate, drift+performance, o eventos de negocio. El retraining debe pasar pruebas y compararse contra champion; nunca desplegar automáticamente solo porque 'ya pasó un mes'.

## Alertas útiles
- schema break: inmediata;
- missing/rango severo: inmediata;
- drift leve: investigar;
- performance debajo de SLA: rollback/retrain;
- cambios de score/alert volume: revisar threshold y upstream.

## Ejercicios
1. Implementa JS divergence para una variable binned.
2. Simula gradual drift por 12 meses y grafica PSI.
3. Introduce schema drift renombrando una columna.
4. Construye dashboard de missing/range/drift por semana.
5. Evalúa AUC y calibration por segmento.
6. Diseña una política de retraining con gates.
7. Investiga Evidently, whylogs, Arize, Fiddler y MLflow monitoring.
